# 5. Évaluation : recherche, réponses, RAGAS

Tous les chiffres de la présentation, recalculés à partir des fichiers de résultats.

_Notebook rejoué avec le code du projet (`graphrag/`, et `app/` du projet vectoriel). Ses sorties sont réelles : il a été exécuté le 10/11/2026 à 03:04 par `scripts/build_notebooks.py`. Aucun appel au LLM n'est fait ici ; Neo4j doit être démarré._

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# candidats de recherche déjà calculés (MultiQuery, HyDE) : aucune requête au LLM
os.environ.setdefault("SEED_CACHE_PATH", str(ROOT / "data" / "cache" / "user_test_candidates.json"))

import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

from graphrag import config
config.use_vector_rag()
DATA = config.VECTOR_RAG_DIR / "data" / "eurlex"
RESULTS = ROOT / "results"
print("Projet :", ROOT)
print("Données du projet vectoriel :", DATA)

Projet : C:\Users\hazem\5eme\GraphRAG-juridique
Données du projet vectoriel : C:\Users\hazem\5eme\Advenced-Rag-Agents-main\Advenced-Rag-Agents-main\data\eurlex


In [2]:
def resume(nom, metrique):
    d = json.loads((RESULTS / nom).read_text(encoding="utf-8"))
    lignes = [{"mode": r["mode"], "sous_ensemble": f"{k} ({s['n']})", metrique: s[metrique]}
              for r in d["results"] for k, s in r["summary"].items()]
    return pd.DataFrame(lignes).pivot(index="sous_ensemble", columns="mode", values=metrique).round(3)

print("60 questions : Recall@5")
resume("eval_graph_20261108_215757.json", "recall@k")

60 questions : Recall@5


mode,advanced,graph,graph_linked,vector
sous_ensemble,,,,
factuelle (34),1.000,1.000,1.000,0.941
global (60),0.858,0.869,0.922,0.811
multi_hop (20),0.625,0.608,0.767,0.583
relationnelle (6),0.833,1.000,1.000,0.833


In [3]:
resume("eval_graph_20261108_215757.json", "mrr")

mode,advanced,graph,graph_linked,vector
sous_ensemble,,,,
factuelle (34),0.859,0.859,0.859,0.712
global (60),0.866,0.873,0.870,0.727
multi_hop (20),0.887,0.885,0.875,0.783
relationnelle (6),0.833,0.917,0.917,0.625


In [4]:
print("17 questions à l'aveugle : Recall@5")
resume("eval_graph_20261110_005139.json", "recall@k")

17 questions à l'aveugle : Recall@5


mode,advanced,graph,graph_linked,vector
sous_ensemble,,,,
A_article_long (5),0.733,0.733,0.900,0.533
B_multi_hop (7),0.440,0.440,0.869,0.440
C_relationnelle (5),0.800,1.000,1.000,0.600
global (17),0.632,0.691,0.917,0.515


## Test des signes

Question par question, `graph_linked` contre la baseline `advanced` (rappel meilleur, moins bon, égal).

In [5]:
from scipy.stats import binomtest

def signes(nom, a="advanced", b="graph_linked"):
    d = json.loads((RESULTS / nom).read_text(encoding="utf-8"))
    rec = {r["mode"]: {x["id"]: x["recall"] for x in r["rows"]} for r in d["results"]}
    mieux = sum(rec[b][q] > rec[a][q] for q in rec[a])
    moins = sum(rec[b][q] < rec[a][q] for q in rec[a])
    return {"mieux": mieux, "moins_bien": moins, "egal": len(rec[a]) - mieux - moins,
            "p_bilateral": round(binomtest(mieux, mieux + moins, 0.5).pvalue, 4)}

pd.DataFrame({"60 questions": signes("eval_graph_20261108_215757.json"),
              "17 questions à l'aveugle": signes("eval_graph_20261110_005139.json")}).T

,mieux,moins_bien,egal,p_bilateral
60 questions,9.0,2.0,49.0,0.0654
17 questions à l'aveugle,9.0,0.0,8.0,0.0039


## Réponses générées

Articles attendus sélectionnés et cités, calculés automatiquement ; verdict issu de la relecture manuelle contre le texte de loi (détail et erreurs dans le README, §9.5).

In [6]:
VERDICT = {  # relecture manuelle : baseline, Graph RAG
    "P1": ("juste", "incomplète"), "P2": ("juste", "juste"), "A2": ("juste", "juste"), "A3": ("juste", "juste, 1 erreur de référence"),
    "B4": ("juste", "juste"), "B5": ("juste, 1 erreur de numéro", "juste"), "B6": ("juste", "juste"),
    "C3": ("juste", "juste, 1 erreur de détail"), "C5": ("juste", "juste"), "C2": ("ne permet pas de répondre", "juste"),
    "D1": ("refus", "refus"), "D2": ("refus", "refus"), "D3": ("refus", "refus"), "D4": ("refus", "refus")}
lignes = []
for nom in ("answers_utilisateur_20261110_010908.json", "answers_utilisateur_20261110_015420.json"):
    for r in json.loads((RESULTS / nom).read_text(encoding="utf-8"))["rows"]:
        n = len(r["gold"])
        lignes.append({"id": r["id"], "question": r["question"][:70],
                       "baseline_selectionnes": f"{len(r['advanced']['gold_selected'])}/{n}",
                       "graph_selectionnes": f"{len(r['graph_linked']['gold_selected'])}/{n}",
                       "verdict_baseline": VERDICT[r["id"]][0], "verdict_graph": VERDICT[r["id"]][1]})
pd.DataFrame(lignes)

,id,question,baseline_selectionnes,graph_selectionnes,verdict_baseline,verdict_graph
0,P1,Une plateforme peut-elle faire de la publicité ciblée à partir de donn,1/1,0/1,juste,incomplète
1,P2,Comment l'analyse d'impact sur les droits fondamentaux de l'AI Act s'a,1/1,1/1,juste,juste
2,A2,Un fournisseur d'IA à haut risque peut-il traiter des données sensible,1/1,1/1,juste,juste
3,B4,Peut-on montrer à un mineur de la publicité fondée sur le profilage ?,1/2,2/2,juste,juste
4,C3,Quels actes du corpus modifient le règlement 2017/2394 ?,2/2,2/2,juste,"juste, 1 erreur de détail"
5,D2,Quel montant d'amende la CNIL a-t-elle infligé à Google en 2022 ?,0/0,0/0,refus,refus
6,A3,Comment les droits du Data Act s'articulent-ils avec les droits d'accè,2/3,3/3,juste,"juste, 1 erreur de référence"
7,B5,À quelles conditions des données personnelles peuvent-elles être réuti,1/2,2/2,"juste, 1 erreur de numéro",juste
8,C5,Quel acte du corpus modifie la directive 2001/29/CE ?,1/1,1/1,juste,juste
9,D3,Combien de temps un cookie peut-il être conservé selon la loi français,0/0,0/0,refus,refus


## Citations entre guillemets

Chaque fragment cité entre « » (coupé aux « … ») est-il présent mot pour mot dans les articles transmis ?

In [7]:
import re
import unicodedata

articles = {a["id"]: a for a in pd.read_json(DATA / "articles.jsonl", lines=True).to_dict("records")}

def norm(s):
    s = unicodedata.normalize("NFKC", s).replace("’", "'").replace(" ", " ")
    return re.sub(r"\s+", " ", s).lower()

data = json.loads((RESULTS / "answers_utilisateur_20261110_010908.json").read_text(encoding="utf-8"))
compte = {}
for mode in data["modes"]:
    total = trouves = 0
    for row in data["rows"]:
        ctx = norm(" ".join(articles[i]["article_title"] + " " + articles[i]["text"] for i in row[mode]["selected"]))
        for q in re.findall(r"«([^«»]{15,})»", row[mode]["answer"]):
            for frag in re.split(r"…|\.\.\.", q):
                f = norm(frag).strip(" ,.;:")
                if len(f) >= 15:
                    total += 1
                    trouves += f in ctx
    compte[mode] = f"{trouves}/{total}"
compte

{'advanced': '32/37', 'graph_linked': '29/34'}

## RAGAS (juge gpt-oss-20b)

Sur 3 questions à contexte court seulement (B4, C3, C5), à cause de la limite Groq de 8 000 tokens par minute.

In [8]:
ragas = json.loads((RESULTS / "ragas_utilisateur_20261110_013124.json").read_text(encoding="utf-8"))
tab = pd.DataFrame(ragas["resultats"]).drop(columns="meme_selection")
display_tab = tab.pivot(index="id", columns="mode", values=["faithfulness", "answer_relevancy", "context_recall"])
display_tab.loc["moyenne"] = display_tab.mean().round(3)
display_tab

faithfulness               ... context_recall             
mode        advanced graph_linked  ...       advanced graph_linked
id                                 ...                            
B4               1.0        1.000  ...            1.0          1.0
C3               1.0        0.833  ...            1.0          1.0
C5               1.0        1.000  ...            1.0          1.0
moyenne          1.0        0.944  ...            1.0          1.0

[4 rows x 6 columns]